In [0]:
%run ./config


✅ Config loaded


In [0]:

# ============================================
# CONFIGURATION - CHANGE FOR TEST/PROD
# ============================================

# Set to True for testing, False for production
TEST_MODE = True  # ← Change to False for production

STORAGE_ACCOUNT = "funddatalakeshantanu"

# Volume path - automatically switches based on TEST_MODE
if TEST_MODE:
    VOLUME_PATH = "/Volumes/workspace/default/test_ingest_volume/"
    CATALOG_NAME = "workspace"
    SCHEMA_NAME = "default"
    print("🧪 TEST MODE ENABLED")
else:
    VOLUME_PATH = "/Volumes/workspace/default/adls_ingest_volume/"
    CATALOG_NAME = "stock_exchange_api_catalog"
    SCHEMA_NAME = "silver"
    print("🏭 PRODUCTION MODE ENABLED")

print(f"✅ Reading from volume: {VOLUME_PATH}")
print(f"✅ Using catalog: {CATALOG_NAME}.{SCHEMA_NAME}")

# Note: ADLS configuration removed - not needed on AWS Serverless compute
# Data will be written to Unity Catalog managed tables instead

# ============================================
# READ JSON FROM VOLUME
# ============================================

from pyspark.sql.functions import col, explode, to_timestamp, from_unixtime, arrays_zip

# Read all JSON files from volume
df = spark.read.option("multiline", "true").json(VOLUME_PATH)

print(f"✅ Data loaded from Volume: {df.count()} file(s)")

# ============================================
# EXTRACT METADATA
# ============================================

# First explode chart.result to get individual stock results
result_df = df.select(explode(col("chart.result")).alias("result"))

meta_df = result_df.select(
    col("result.meta.symbol").alias("symbol"),
    col("result.meta.regularMarketPrice").alias("current_price"),
    col("result.meta.regularMarketDayHigh").alias("day_high"),
    col("result.meta.regularMarketDayLow").alias("day_low"),
    col("result.meta.regularMarketVolume").alias("volume"),
    to_timestamp(from_unixtime(col("result.meta.regularMarketTime"))).alias("market_time"),
    col("result.meta.currency").alias("currency"),
    col("result.meta.exchangeName").alias("exchange")
)

print("✅ Metadata extracted")

# ============================================
# EXTRACT QUOTE DATA
# ============================================

# Use arrays_zip to combine parallel arrays, then explode
quote_df = result_df.select(
    col("result.meta.symbol").alias("symbol"),
    explode(arrays_zip(
        col("result.timestamp"),
        col("result.indicators.quote.open")[0],
        col("result.indicators.quote.high")[0],
        col("result.indicators.quote.low")[0],
        col("result.indicators.quote.close")[0],
        col("result.indicators.quote.volume")[0]
    )).alias("data")
)

quote_df = quote_df.select(
    col("symbol"),
    to_timestamp(from_unixtime(col("data.timestamp"))).alias("timestamp"),
    col("data.`1`").alias("open"),
    col("data.`2`").alias("high"),
    col("data.`3`").alias("low"),
    col("data.`4`").alias("close"),
    col("data.`5`").alias("volume")
)

print(f"✅ Quote data extracted: {quote_df.count()} records")

# ============================================
# WRITE TO UNITY CATALOG DELTA TABLES
# ============================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

meta_df.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.stock_meta")
quote_df.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.stock_quotes")

print(f"✅ Silver tables created!")
print(f"Metadata: {meta_df.count()} records")
print(f"Quotes: {quote_df.count()} records")
print(f"✅ Tables saved to {CATALOG_NAME}.{SCHEMA_NAME}")

# ============================================
# VERIFY
# ============================================

print("\n📊 Metadata Table:")
display(spark.sql(f"SELECT * FROM {CATALOG_NAME}.{SCHEMA_NAME}.stock_meta"))

print("\n📊 Quotes Table (first 5 rows):")
display(spark.sql(f"SELECT * FROM {CATALOG_NAME}.{SCHEMA_NAME}.stock_quotes").limit(5))

print("\n✅ Silver layer complete!")

🧪 TEST MODE ENABLED
✅ Reading from volume: /Volumes/workspace/default/test_ingest_volume/
✅ Using catalog: workspace.default
✅ Data loaded from Volume: 5 file(s)
✅ Metadata extracted
✅ Quote data extracted: 1955 records
✅ Silver tables created!
Metadata: 5 records
Quotes: 1955 records
✅ Tables saved to workspace.default

📊 Metadata Table:


symbol,current_price,day_high,day_low,volume,market_time,currency,exchange
AAPL,308.91,310.69,300.0,127398021,2026-07-31T20:00:01.000Z,USD,NMS
GOOG,356.65,358.88,339.17,30351093,2026-07-31T20:00:01.000Z,USD,NMS
MSFT,464.72,466.84,449.552,56468734,2026-07-31T20:00:01.000Z,USD,NMS
NVDA,200.75,201.97,194.95,139261166,2026-07-31T20:00:01.000Z,USD,NMS
TSLA,311.21,315.5,301.97,35616593,2026-07-31T20:00:00.000Z,USD,NMS



📊 Quotes Table (first 5 rows):


symbol,timestamp,open,high,low,close,volume
AAPL,2026-07-31T13:30:00.000Z,304.80999755859375,306.8999938964844,303.6400146484375,304.04998779296875,9663898
AAPL,2026-07-31T13:31:00.000Z,304.010009765625,305.010009765625,301.8299865722656,302.0299987792969,1704033
AAPL,2026-07-31T13:32:00.000Z,301.8999938964844,302.7875061035156,301.3414001464844,302.0299987792969,2027555
AAPL,2026-07-31T13:33:00.000Z,302.07000732421875,302.57000732421875,301.07000732421875,301.8216857910156,1823229
AAPL,2026-07-31T13:34:00.000Z,301.8299865722656,302.39990234375,300.54998779296875,302.1099853515625,1758609



✅ Silver layer complete!
